# Module 3 — Data Quality Assessment & Cleaning
Data Engineer role. Lecture 7: real data is *incomplete*, *noisy* and *inconsistent*; preparation must remove
errors **without adding bias** ("the more guessing you have to do, the further away from the real data").

1. Quality assessment by dimension: completeness, uniqueness, validity, consistency, referential integrity
2. Missing-value analysis (what does each missing value *mean*?)
3. Outliers: course-specific IQR fences (univariate) and bot-like accounts (multi-rule, multivariate)
4. Cleaning with a full operation log
5. Before/after comparisons

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src")) if (Path.cwd().parent / "src").exists() else sys.path.insert(0, "src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from edtech import config as C, oulad, cleaning as K, features as F, utils as U, viz as V
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)

t = oulad.read_raw()
enr = pd.read_parquet(C.INTERIM / "oulad_enrollments_raw.parquet")
sessions = pd.read_parquet(C.SYNTHETIC / "synthetic_login_sessions.parquet")
subs = pd.read_parquet(C.SYNTHETIC / "synthetic_submission_events.parquet")

## 1. Quality scorecard

In [2]:
keys = {"courses": ["code_module", "code_presentation"], "assessments": ["id_assessment"],
        "studentInfo": C.KEY, "studentRegistration": C.KEY, "studentAssessment": ["id_assessment", "id_student"],
        "vle": ["id_site"], "synthetic_login_sessions": ["session_id"],
        "synthetic_submission_events": ["id_assessment", "id_student"]}
tables = {**t, "synthetic_login_sessions": sessions, "synthetic_submission_events": subs}
score = K.quality_scorecard(tables, keys)
U.save_table(score, "m3_quality_scorecard"); score

,table,rows,columns,completeness_pct,missing_cells,exact_duplicate_rows,duplicate_keys,key
0,courses,22,3,100.000,0,0,0,"code_module,code_presentation"
1,assessments,206,6,99.110,11,0,0,id_assessment
2,studentInfo,32593,12,99.716,1111,0,0,"id_student,code_module,code_presentation"
3,studentRegistration,32593,5,86.153,22566,0,0,"id_student,code_module,code_presentation"
4,studentAssessment,173912,5,99.980,173,0,0,"id_assessment,id_student"
5,vle,6364,6,72.538,10486,0,0,id_site
6,synthetic_login_sessions,2709570,14,99.786,81158,21505,21505,session_id
7,synthetic_submission_events,167044,9,100.000,0,0,0,"id_assessment,id_student"


In [3]:
ref = K.referential_checks(t, sessions); U.save_table(ref, "m3_referential_integrity"); ref

,relationship,child_keys,orphan_keys,integrity_pct
0,studentRegistration -> studentInfo,32593,0,100.0
1,studentAssessment -> assessments,188,0,100.0
2,studentInfo -> courses,22,0,100.0
3,vle -> courses,22,0,100.0
4,synthetic sessions -> studentInfo,31807,0,100.0


In [4]:
val = K.validity_checks_sessions(sessions)
val["violation_pct"] = (100 * val.violations / val.denominator).round(3)
U.save_table(val, "m3_validity_sessions"); val

,rule,violations,denominator,violation_pct
0,session_minutes < 0,5412,2709570,0.200
1,session_minutes > 1440 (longer than a day),2724,2709570,0.101
2,videos_completed > videos_started,8137,2709570,0.300
3,login_hour outside 0..23,0,2709570,0.000
4,device_type not in canonical vocabulary,214488,2709570,7.916
5,device_type unmappable after standardisation,0,2709570,0.000
6,forum_replies > forum_posts,0,2709570,0.000


In [5]:
cons = pd.DataFrame([
    ("imd_band uses two spellings for one decile ('10-20' vs 'x-y%')", int(t["studentInfo"].imd_band.eq("10-20").sum())),
    ("Withdrawn outcome without unregistration date, or date without Withdrawn",
     int(((enr.final_result == "Withdrawn") != enr.date_unregistration.notna()).sum())),
    ("Submission after course end (excluded from current-behaviour features)",
     int((t["studentAssessment"].merge(t["assessments"], on="id_assessment").merge(t["courses"], on=["code_module", "code_presentation"])
          .eval("date_submitted > module_presentation_length")).sum())),
    ("First recorded activity before registration date", int((enr.first_active_day < enr.date_registration).sum())),
    ("Synthetic device_type: 17 spellings of 3 classes", int(val.loc[val.rule.str.startswith("device_type not"), "violations"].iloc[0])),
], columns=["consistency check", "records"])
U.save_table(cons, "m3_consistency"); cons

,consistency check,records
0,imd_band uses two spellings for one decile ('10-20' vs 'x-y%'),3516
1,"Withdrawn outcome without unregistration date, or date without Withdrawn",102
2,Submission after course end (excluded from current-behaviour features),85
3,First recorded activity before registration date,0
4,Synthetic device_type: 17 spellings of 3 classes,214488


**Summary of quality findings.** OULAD keys are unique and all foreign keys resolve (100% integrity); the
issues are an inconsistent IMD label, administratively missing dates, 787k identical clickstream rows whose
status cannot be decided (no event id), and cross-field disagreements. The synthetic log carries the realistic
defects of an operational system: double-logged rows, inconsistent device labels, missing and impossible
durations, completed-without-started videos and bot traffic.

## 2. Missing values — mechanism and treatment

In [6]:
miss = pd.concat([K.missing_profile(v, k) for k, v in tables.items()], ignore_index=True)
meaning = {
    "date": "Exam deadline not published (structural, MNAR) → keep NA; exams excluded from timeliness",
    "imd_band": "Deprivation band unknown (MAR on region) → explicit 'Unknown' + flag",
    "date_registration": "Registration day not recorded → NA + flag; only affects days-to-first-login",
    "date_unregistration": "Learner did not unregister (structural, not an error) → keep NA",
    "score": "Mark not recorded → excluded from mean; non-submission counted as 0 in avg_score_scheduled",
    "week_from": "Resource not planned for a specific week (structural) → not used",
    "week_to": "Resource not planned for a specific week (structural) → not used",
    "session_minutes": "Session not closed / logging gap (MCAR by design) → learner-median imputation + flag",
    "device_type": "Client did not report device → 'unknown' category",
}
miss["meaning_and_treatment"] = miss.column.map(meaning)
U.save_table(miss, "m3_missing_values"); miss

,table,column,missing,missing_pct,meaning_and_treatment
0,assessments,date,11,5.340,"Exam deadline not published (structural, MNAR) → keep NA; exams excluded from timeliness"
1,studentInfo,imd_band,1111,3.409,Deprivation band unknown (MAR on region) → explicit 'Unknown' + flag
2,studentRegistration,date_registration,45,0.138,Registration day not recorded → NA + flag; only affects days-to-first-login
3,studentRegistration,date_unregistration,22521,69.098,"Learner did not unregister (structural, not an error) → keep NA"
4,studentAssessment,score,173,0.099,Mark not recorded → excluded from mean; non-submission counted as 0 in avg_score_sched...
5,vle,week_from,5243,82.385,Resource not planned for a specific week (structural) → not used
6,vle,week_to,5243,82.385,Resource not planned for a specific week (structural) → not used
7,synthetic_login_sessions,device_type,27112,1.001,Client did not report device → 'unknown' category
8,synthetic_login_sessions,session_minutes,54046,1.995,Session not closed / logging gap (MCAR by design) → learner-median imputation + flag


Lecture 7 warns against automatic replacement of missing values: most OULAD gaps are **structural** (a learner who
never withdrew has no withdrawal date), so filling them would invent facts. Only `session_minutes` — a random
logging gap — is imputed, using the learner's own median (the "same-class mean" method) with an indicator column.

## 3. Outliers
### 3a. Univariate: course-specific IQR fences on clicks
Click volume depends on course design, so fences are computed within each module-presentation.

In [7]:
log = K.CleaningLog()
enr_clean = K.clean_oulad(t, enr, log)
o = enr_clean.groupby(["code_module", "code_presentation"]).agg(enrollments=("id_student", "size"),
                                                                 extreme_click_outliers=("clicks_extreme_outlier", "sum"))
print("Extreme click outliers:", int(enr_clean.clicks_extreme_outlier.sum()))
fig, ax = plt.subplots(figsize=(10, 4))
x = enr_clean[enr_clean.total_clicks > 0]
ax.boxplot([np.log1p(x.loc[x.code_module == m, "total_clicks"]) for m in sorted(x.code_module.unique())],
           tick_labels=sorted(x.code_module.unique()), flierprops={"markersize": 2, "alpha": 0.3})
ax.set(title="log(1 + clicks) by module — outliers are high but plausible activity", ylabel="log(1 + total clicks)")
V.save(fig, "m3_click_outliers"); plt.show()

Extreme click outliers: 551


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13636\2268296436.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  V.save(fig, "m3_click_outliers"); plt.show()


These learners are genuinely very active (they pass at higher rates); deleting them would remove the most engaged
learners. Decision: **keep and flag**, and reduce leverage with a power transform + winsorisation inside the
feature pipeline (Lecture 7 options: "do nothing", "enforce bounds", "binning/transform").

### 3b. Multivariate: bot-like accounts
No single rule is reliable: very active humans also have many sessions. We combine four signals and flag an
enrollment when **at least two** fire:

| Rule | Signal |
|---|---|
| R1 | session count > Q3 + 3·IQR within the course (extreme outlier) |
| R2 | > 30% of sessions shorter than 1 minute |
| R3 | > 10% of sessions from an automation user agent (python-requests, curl, bot…) |
| R4 | login-hour entropy > 4.2 bits (humans sleep; uniform 24 h activity ≈ 4.58 bits) |

In [8]:
s_dedup = sessions.drop_duplicates()
sig = K.bot_signals(s_dedup, enr)
rules = ["r1_extreme_session_count", "r2_sub_minute_sessions", "r3_automation_user_agent", "r4_uniform_hours"]
truth = pd.DataFrame(json.loads((C.SYNTHETIC / "_injected_defects.json").read_text())["bot_enrollments"]).assign(is_bot=1)
m = sig.merge(truth, on=C.KEY, how="left").fillna({"is_bot": 0})
rule_eval = []
for r in rules + ["bot_flag"]:
    tp = int(((m[r] == 1) & (m.is_bot == 1)).sum()); fp = int(((m[r] == 1) & (m.is_bot == 0)).sum()); fn = int(((m[r] == 0) & (m.is_bot == 1)).sum())
    rule_eval.append({"rule": r, "flagged": tp + fp, "true_bots": tp, "false_positives": fp,
                      "precision": round(tp / max(tp + fp, 1), 3), "recall": round(tp / max(tp + fn, 1), 3)})
rule_eval = pd.DataFrame(rule_eval); U.save_table(rule_eval, "m3_bot_rule_evaluation"); rule_eval

,rule,flagged,true_bots,false_positives,precision,recall
0,r1_extreme_session_count,221,109,112,0.493,0.956
1,r2_sub_minute_sessions,124,114,10,0.919,1.000
2,r3_automation_user_agent,114,114,0,1.000,1.000
3,r4_uniform_hours,114,114,0,1.000,1.000
4,bot_flag,114,114,0,1.000,1.000


In [9]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
hum, bot = m[m.bot_flag == 0], m[m.bot_flag == 1]
ax[0].scatter(hum.sessions, hum.hour_entropy_bits, s=4, alpha=0.2, color=V.CATEGORICAL[0], label="human")
ax[0].scatter(bot.sessions, bot.hour_entropy_bits, s=14, color=V.CATEGORICAL[7], label="flagged bot")
ax[0].axhline(4.2, color=V.TEXT2, ls="--", lw=1); ax[0].set_xscale("log")
ax[0].set(title="Session count vs login-hour entropy", xlabel="sessions (log scale)", ylabel="entropy of login hour (bits)"); ax[0].legend()
ax[1].scatter(hum.sessions, hum.share_sub_minute, s=4, alpha=0.2, color=V.CATEGORICAL[0], label="human")
ax[1].scatter(bot.sessions, bot.share_sub_minute, s=14, color=V.CATEGORICAL[7], label="flagged bot")
ax[1].axhline(0.30, color=V.TEXT2, ls="--", lw=1); ax[1].set_xscale("log")
ax[1].set(title="Session count vs share of sub-minute sessions", xlabel="sessions (log scale)", ylabel="share < 1 minute"); ax[1].legend()
V.save(fig, "m3_bot_detection"); plt.show()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13636\1368715078.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  V.save(fig, "m3_bot_detection"); plt.show()


R1 alone would also flag dozens of genuinely heavy human learners (false positives); requiring two independent
signals isolates the automated accounts with no false positives against the injected ground truth. Treatment:
**drop only the automated sessions** of flagged enrollments (their human sessions remain), instead of deleting
the learner.

## 4. Cleaning

In [10]:
sess_clean, signals = K.clean_sessions(sessions, enr_clean, log)
subs_clean = K.clean_submissions(subs, log)
bot_eval = K.evaluate_bot_detection(signals)
cl = log.frame(); U.save_table(cl, "m3_cleaning_log"); cl

,source,issue,rule,action,affected,denominator,affected_pct,justification
0,OULAD,inconsistent categorical label,imd_band == '10-20',relabel to '10-20%',3516,32593,10.788,"All other IMD bands use the 'x-y%' format; same decile, two spellings."
1,OULAD,missing value,imd_band is null,category 'Unknown' + flag,1111,32593,3.409,Deprivation band is not imputable from behaviour; explicit category avoids bias (Lectu...
2,OULAD,missing value,date_registration is null,keep NA + flag,45,32593,0.138,"Registration day is unknown, not zero; only used for days-to-first-login, which become..."
3,OULAD,cross-field inconsistency,Withdrawn XOR date_unregistration present,flag only,102,32593,0.313,"Administrative fields disagree; neither can be verified, so no value is invented."
4,OULAD,outlier,total_clicks > Q3 + 3 IQR within course,flag; Yeo-Johnson + winsorising in features,551,32593,1.691,Very active learners are real; Yeo-Johnson transform + winsorising limit leverage (Lec...
5,OULAD,possible duplicate events,identical studentVle rows,retain (documented sensitivity),787170,10655280,7.388,OULAD has no event id; identical daily rows may be distinct interactions. Removing the...
6,Synthetic sessions,duplicate rows,all columns identical,drop,21505,2709570,0.794,Same session_id and identical payload = logging retry.
7,Synthetic sessions,duplicate key,session_id repeated with different payload,drop later copy,0,2688065,0.000,Primary key must be unique.
8,Synthetic sessions,inconsistent categorical labels,device_type spelling/synonyms,"map to {desktop, mobile, tablet}",212852,2688065,7.918,"e.g. 'PC', ' desktop', 'DESKTOP' are the same device class."
9,Synthetic sessions,missing value,device_type is null,category 'unknown',26880,2688065,1.000,Device cannot be inferred reliably from a free-text user agent.


In [11]:
enr_clean.to_parquet(C.INTERIM / "oulad_enrollments_clean.parquet", index=False)
sess_clean.to_parquet(C.INTERIM / "sessions_clean.parquet", index=False)
subs_clean.to_parquet(C.INTERIM / "submissions_clean.parquet", index=False)
cl.to_csv(C.INTERIM / "cleaning_log.csv", index=False)
print(len(sessions), "->", len(sess_clean), "session rows")

2709570 -> 2580034 session rows


## 5. Before / after comparison

In [12]:
ba = K.before_after(sessions, sess_clean, ["session_minutes", "videos_started", "videos_completed", "forum_posts"])
U.save_table(ba.round(3), "m3_before_after_sessions"); ba.round(2)

,column,stage,rows,missing,mean,median,std,min,p99,max
0,session_minutes,before,2709570,54046,29.57,14.5,184.18,-541.2,162.0,8988.7
1,session_minutes,after,2580034,0,25.13,15.2,32.74,0.9,156.6,1434.2
2,videos_started,before,2709570,0,0.51,0.0,0.99,0.0,4.0,19.0
3,videos_started,after,2580034,0,0.53,0.0,1.00,0.0,5.0,19.0
4,videos_completed,before,2709570,0,0.36,0.0,0.80,0.0,4.0,17.0
5,videos_completed,after,2580034,0,0.37,0.0,0.80,0.0,4.0,17.0
6,forum_posts,before,2709570,0,0.09,0.0,0.42,0.0,2.0,14.0
7,forum_posts,after,2580034,0,0.10,0.0,0.43,0.0,2.0,14.0


In [13]:
dev = pd.DataFrame({"before": sessions.device_type.fillna("<missing>").value_counts(),
                    "after": sess_clean.device_type.value_counts()}).fillna(0).astype(int)
U.save_table(dev.reset_index(names="device_label"), "m3_device_before_after"); dev

,before,after
device_type,,
desktop,20720,0
<missing>,27112,0
DESKTOP,20837,0
Desktop,21021,0
MOBILE,15063,0
Mobile,15118,0
PC,20889,0
Smartphone,15250,0
TABLET,8568,0


In [14]:
agg_before = F.session_aggregates(sessions, enr)
agg_after = F.session_aggregates(sess_clean, enr_clean)
cmp_ = agg_before.merge(agg_after, on=C.KEY, suffixes=("_before", "_after"), how="outer")
for c in ["logins_total", "mean_session_minutes", "videos_completed", "videos_started"]:
    cmp_[[c + "_before", c + "_after"]] = cmp_[[c + "_before", c + "_after"]].fillna(0)
w = enr_clean.set_index(C.KEY)["observation_weeks"]
cmp_ = cmp_.join(w, on=C.KEY)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
ax[0].hist(np.log10(1 + cmp_.logins_total_before / cmp_.observation_weeks), bins=80, histtype="step", lw=2, label="before", color=V.CATEGORICAL[7])
ax[0].hist(np.log10(1 + cmp_.logins_total_after / cmp_.observation_weeks), bins=80, histtype="step", lw=2, label="after", color=V.CATEGORICAL[0])
ax[0].set(title="Logins per week", xlabel="log10(1 + logins/week)", ylabel="enrollments", yscale="log"); ax[0].legend()
vb = (cmp_.videos_completed_before / cmp_.videos_started_before.replace(0, np.nan)).dropna()
va = (cmp_.videos_completed_after / cmp_.videos_started_after.replace(0, np.nan)).dropna()
ax[1].hist(vb, bins=60, histtype="step", lw=2, label="before", color=V.CATEGORICAL[7], range=(0, 2))
ax[1].hist(va, bins=60, histtype="step", lw=2, label="after", color=V.CATEGORICAL[0], range=(0, 2))
ax[1].axvline(1, color=V.TEXT2, ls="--", lw=1)
ax[1].set(title="Raw video completion ratio", xlabel="completed / started", ylabel="enrollments"); ax[1].legend()
ax[2].hist(sessions.session_minutes.clip(-20, 120).dropna(), bins=70, histtype="step", lw=2, label="before", color=V.CATEGORICAL[7])
ax[2].hist(sess_clean.session_minutes.clip(-20, 120), bins=70, histtype="step", lw=2, label="after", color=V.CATEGORICAL[0])
ax[2].set(title="Session minutes (display clipped −20..120)", xlabel="minutes", ylabel="sessions"); ax[2].legend()
fig.tight_layout(); V.save(fig, "m3_before_after"); plt.show()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13636\542685230.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); V.save(fig, "m3_before_after"); plt.show()


In [15]:
imd = pd.DataFrame({"before": t["studentInfo"].imd_band.fillna("<missing>").value_counts(),
                    "after": enr_clean.imd_band.value_counts()}).fillna(0).astype(int)
U.save_table(imd.reset_index(names="imd_band"), "m3_imd_before_after"); imd

,before,after
imd_band,,
0-10%,3311,3311
10-20,3516,0
10-20%,0,3516
20-30%,3654,3654
30-40%,3539,3539
40-50%,3256,3256
50-60%,3124,3124
60-70%,2905,2905
70-80%,2879,2879


In [16]:
summary = {
    "session_rows_before": len(sessions), "session_rows_after": len(sess_clean),
    "enrollments_before": len(enr), "enrollments_after": len(enr_clean),
    "bot_detection": bot_eval, "flagged_bot_enrollments": int(signals.bot_flag.sum()),
    "r1_only_false_positives": int(rule_eval.loc[rule_eval.rule == "r1_extreme_session_count", "false_positives"].iloc[0]),
    "imd_relabelled": int(t["studentInfo"].imd_band.eq("10-20").sum()), "imd_missing": int(t["studentInfo"].imd_band.isna().sum()),
    "duplicate_session_rows": int(sessions.duplicated().sum()),
    "click_extreme_outliers": int(enr_clean.clicks_extreme_outlier.sum()),
    "scorecard": score.to_dict("records"),
}
U.save_metrics("m3", summary); summary

{'session_rows_before': 2709570,
 'session_rows_after': 2580034,
 'enrollments_before': 32593,
 'enrollments_after': 32593,
 'bot_detection': {'true_positive': 114,
  'false_positive': 0,
  'false_negative': 0,
  'precision': 1.0,
  'recall': 1.0},
 'flagged_bot_enrollments': 114,
 'r1_only_false_positives': 112,
 'imd_relabelled': 3516,
 'imd_missing': 1111,
 'duplicate_session_rows': 21505,
 'click_extreme_outliers': 551,
 'scorecard': [{'table': 'courses',
   'rows': 22,
   'columns': 3,
   'completeness_pct': 100.0,
   'missing_cells': 0,
   'exact_duplicate_rows': 0,
   'duplicate_keys': 0,
   'key': 'code_module,code_presentation'},
  {'table': 'assessments',
   'rows': 206,
   'columns': 6,
   'completeness_pct': 99.11,
   'missing_cells': 11,
   'exact_duplicate_rows': 0,
   'duplicate_keys': 0,
   'key': 'id_assessment'},
  {'table': 'studentInfo',
   'rows': 32593,
   'columns': 12,
   'completeness_pct': 99.716,
   'missing_cells': 1111,
   'exact_duplicate_rows': 0,
   'dup

## Conclusions
* **No enrollment is removed** (32,593 before and after): every cleaning action works on values or on
  automated/duplicate *events*, so learner coverage — and therefore the at-risk population — is preserved.
* The cleaned synthetic log loses ≈ 5% of its rows: double-logged rows and automated bot sessions.
* Missingness is preserved where it carries meaning (structural dates, unknown IMD) and imputed only where it is
  a random logging gap; every imputation is flagged.
* Remaining uncertainty: 787k identical OULAD click rows are kept (no event id to prove duplication; dropping them
  would change window clicks by ≈ 3%, a documented sensitivity).